# Constructing Machinery Representation

The **machinery representation** is the lowest level of the three: it is the machine's own program, a
lattice of macronodes each carrying the four homodyne angles it is measured at, plus the displacements and
the feedforward coefficients. There are no operations here — an operation *is* a choice of angles, and
{ref}`Graph-representation gates <sec:gates-in-graph-repr>` lists the angles each one becomes.

You would write one to drive the machine directly: to use a set of angles no operation in
{py:mod}`claveles.graph.ops` produces, to set the feedforward coefficients yourself, or to read a macronode at
micronode `a` or `c`, which the graph representation cannot name. Otherwise let
{py:meth}`~claveles.machinery.MachineryRepr.from_graph_repr` or the client produce it.

A representation is constructed by specifying the following parameters.

| Parameter | Setter     | Getter | Description |
| --------- | ---------- | ------ | ----------- |
| `n_local_macronodes` | {py:meth}`~claveles.machinery.MachineryRepr.__init__` | {py:attr}`~claveles.machinery.MachineryRepr.n_local_macronodes` | The number of macronodes per step. |
| `n_steps` | {py:meth}`~claveles.machinery.MachineryRepr.__init__` | {py:attr}`~claveles.machinery.MachineryRepr.n_steps` | The number of steps. |
| `readout_macronode_indices` | {py:attr}`~claveles.machinery.MachineryRepr.readout_macronode_indices` | {py:attr}`~claveles.machinery.MachineryRepr.readout_macronode_indices` | Set of macronode indices to get measurement results. |
| `homodyne_angles` | {py:meth}`~claveles.machinery.MachineryRepr.set_homodyne_angle` | {py:meth}`~claveles.machinery.MachineryRepr.get_homodyne_angle` | List of four measurement angles in each macronode. The setter and getter handle one element at a time. |
| `displacements_k_minus_1` | {py:meth}`~claveles.machinery.MachineryRepr.set_displacement_k_minus_1` | {py:meth}`~claveles.machinery.MachineryRepr.get_displacement_k_minus_1` | Displacement applied to the mode teleporting to the macronode `k` from the macronode `k - 1`. |
| `displacements_k_minus_n` | {py:meth}`~claveles.machinery.MachineryRepr.set_displacement_k_minus_n` | {py:meth}`~claveles.machinery.MachineryRepr.get_displacement_k_minus_n` | Displacement applied to the mode teleporting to the macronode `k` from the macronode `k - n_local_macronodes`. |
| `ff_coeff_matrix_k_plus_1` | {py:attr}`~claveles.machinery.MachineryRepr.ff_coeff_matrix_k_plus_1` | {py:attr}`~claveles.machinery.MachineryRepr.ff_coeff_matrix_k_plus_1` | Feedforward coefficient matrix from the macronode `k` to the macronode `k + 1`. |
| `ff_coeff_matrix_k_plus_n` | {py:attr}`~claveles.machinery.MachineryRepr.ff_coeff_matrix_k_plus_n` | {py:attr}`~claveles.machinery.MachineryRepr.ff_coeff_matrix_k_plus_n` | Feedforward coefficient matrix from the macronode `k` to the macronode `k + n_local_macronodes`. |

By default, the feedforward coefficient matrix is automatically calculated from the homodyne angles.

In [ ]:
# Draw the figures on this page as SVG so that they stay sharp at any zoom level.
# The cell itself is hidden from the built page by the `remove-cell` tag.
%config InlineBackend.figure_formats = ["svg"]

## Creating a machinery representation

In [ ]:
from claveles.machinery import MachineryRepr
from claveles.machinery.visualize import make_figure

m = MachineryRepr(
    n_local_macronodes=3,
    n_steps=4,
)

make_figure(m, readout_edge_color="red");

### Setting macronodes to readout

In [ ]:
m.readout_macronode_indices = {6, 8, 10}

make_figure(m, readout_edge_color="red");

### Setting homodyne angles

In [ ]:
from math import pi

from claveles.machinery.macronode_angle import MacronodeAngle

m.set_homodyne_angle(4, MacronodeAngle(0, pi / 2, 0, pi / 2))
m.set_homodyne_angle(5, MacronodeAngle(pi / 2, 0, pi / 2, 0))
m.set_homodyne_angle(7, MacronodeAngle(0, pi / 2, 0, pi / 2))
m.set_homodyne_angle(10, MacronodeAngle(pi / 2, pi / 2, pi / 2, pi / 2))

make_figure(m, readout_edge_color="red");

### Adding displacements

In [ ]:
m.set_displacement_k_minus_1(4, (1.0, -1.0))

make_figure(m, readout_edge_color="red");

### Setting the generation method of the feedforward coefficient matrix

You can specify how the feedforward coefficient matrix is generated, as shown in the code below.
The available methods are defined in the enum class {py:class}`~claveles.machinery.program.FFCoeffMatrixGenerationMethods`.

Currently, you can choose from the following options:

- {py:attr}`~claveles.machinery.program.FFCoeffMatrixGenerationMethods.ZERO_FILLED`: Initializes the matrix
  with all zeros.
- {py:attr}`~claveles.machinery.program.FFCoeffMatrixGenerationMethods.FROM_MACRONODE_ANGLE`: Computes the
  matrix based on macronode measurement angles.

In [ ]:
from claveles.machinery.program import FFCoeffMatrixGenerationMethods

m.ff_coeff_matrix_k_plus_n.generation_methods[1] = FFCoeffMatrixGenerationMethods.ZERO_FILLED
m.ff_coeff_matrix_k_plus_n.generation_methods[4] = FFCoeffMatrixGenerationMethods.FROM_MACRONODE_ANGLE

### Adding feedforward

Measured variables used for feedforward are created using the {py:class}`~claveles.machinery.macronode_angle.MeasuredVariable` class.  
This class takes two arguments:

- The first argument is the **macronode index**.
- The second argument is which **micronode** within the macronode, named with {py:class}`~claveles.micronode.Micronode`: {py:attr}`Micronode.A <claveles.micronode.Micronode.A>`, {py:attr}`Micronode.B <claveles.micronode.Micronode.B>`, {py:attr}`Micronode.C <claveles.micronode.Micronode.C>` or {py:attr}`Micronode.D <claveles.micronode.Micronode.D>`.

The micronode is named rather than numbered because the graph representation reads only two of them (b and d), so the two layers would number them differently; both take a {py:class}`~claveles.micronode.Micronode`.

All four are available here. The machinery representation measures a macronode at four homodyne angles, and a feedforward may read any of the four micronodes; {py:func}`~claveles.machinery.visualize.make_figure` draws the arrow from the micronode the variable names. The figure below reads `b` of macronode 2 and `c` of macronode 3, and their arrows leave from different points.

```{note}
A graph macronode reports the values measured at `b` and `d` only, so a feedforward reading `a` or `c` has no graph form. {py:meth}`~claveles.machinery.MachineryRepr.to_graph_repr` raises on such a representation rather than folding `a` onto `b` and `c` onto `d`, which would move the arrow to a micronode the program never named. This does not restrict execution or visualization, neither of which goes through that conversion.
```

In [ ]:
from claveles.feedforward import feedforward
from claveles.machinery.macronode_angle import MacronodeAngle, MeasuredVariable
from claveles.micronode import Micronode


@feedforward
def f1(x: float) -> float:
    return x * x


@feedforward
def f2(x: float) -> float:
    return x * x * x


x = MeasuredVariable(2, Micronode.B)  # macronode 2, micronode b
y = MeasuredVariable(3, Micronode.C)  # macronode 3, micronode c
m.set_displacement_k_minus_n(4, (f1(x), f2(y)))
m.set_homodyne_angle(4, MacronodeAngle(f1(x), f2(y), 0, pi / 2))
m.set_homodyne_angle(8, MacronodeAngle(pi / 2, 0, f1(x), f2(y)))

make_figure(m, readout_edge_color="red");

## Converting graph representation to machinery representation

You can construct a machinery representation from a graph representation using the {py:meth}`~claveles.machinery.MachineryRepr.from_graph_repr` method.

In [ ]:
from math import pi

from claveles.feedforward import feedforward
from claveles.graph import GraphRepr, Wiring, ops
from claveles.graph.constant import BLANK_MODE
from claveles.machinery import MachineryRepr


@feedforward
def add_pi(x: float) -> float:
    from math import pi

    return x + pi


@feedforward
def square(x: float) -> float:
    return x * x


@feedforward
def cube(x: float) -> float:
    return x * x * x


graph = GraphRepr(5, 4)
graph.place_operation(ops.Initialization((0, 1), 0.0, (0, BLANK_MODE)))
graph.place_operation(ops.Initialization((1, 0), 0.0, (BLANK_MODE, 1)))
graph.place_operation(ops.Initialization((2, 0), 0.0, (BLANK_MODE, 2)))
graph.place_operation(ops.BeamSplitter((1, 1), 0.0, 0.0, displacement_k_minus_n=(1.0, -1.0), swap=True))
graph.place_operation(ops.TwoModeShear((2, 1), 0.0, 0.0, displacement_k_minus_1=(-1.0, 1.0), swap=True))
graph.place_operation(ops.PhaseRotation((3, 1), pi / 2, swap=True))
graph.place_operation(ops.Measurement((1, 2), 0.0))  # Measure mode 0
m0 = graph.get_mode_measured_value(0)
graph.place_operation(Wiring((2, 2), swap=True))
graph.place_operation(ops.ControlledZ((3, 2), 0.0, swap=False))
graph.place_operation(ops.Measurement((4, 2), 0.0))  # Measure mode 1
m1 = graph.get_mode_measured_value(1)
graph.place_operation(
    ops.ShearPInvariant((3, 3), square(m0), displacement_k_minus_n=(add_pi(m0), cube(m1)), swap=True)
)
graph.place_operation(ops.Measurement((4, 3), 0.0))  # Measure mode 2

m = MachineryRepr.from_graph_repr(graph)

print("#local macronodes         =", m.n_local_macronodes)
print("#steps                    =", m.n_steps)
print("homodyne angles           =", [m.get_homodyne_angle(i) for i in range(m.n_total_macronodes)])
print("readout_macronode_indices =", m.readout_macronode_indices)
print("displacement k-1->k       =", m.displacements_k_minus_1)
print("displacement k-N->k       =", m.displacements_k_minus_n)
print("ff coeff matrix k->k+1    = ", m.ff_coeff_matrix_k_plus_1)
print("ff coeff matrix k->k+N    = ", m.ff_coeff_matrix_k_plus_n)

make_figure(m, readout_edge_color="red");

## Initial states

Every mode in this representation starts as a **squeezed state**, produced by an initialization
macronode; {ref}`the graph.ops.Initialization entry <sec:gate-graph-repr-initialization>` explains why a measurement macronode and an
initialization macronode are the same thing.

Unlike the circuit representation, which lets you place a state on a mode (see
{ref}`sec:circuit-placing-states`), this representation has no way to express a state at all. What can be
set, and where:

| | Set by |
| --- | --- |
| **Squeezing level** | Not expressible here. On the simulator it is the client option {py:attr}`~claveles.client.simulator_client.SimulatorClient.resource_squeezing_level`; on the machine it is fixed by the hardware. |
| **Squeezing angle** | The four measurement angles of the initialization macronode, which have to be equal; that shared angle $\theta$ initializes the mode with {ref}`squeezing angle <sec:squeezing-level-and-squeezing-angle>` $\theta + \pi/2$. |

```{seealso}
{ref}`sec:execution-resource-and-initialized-states` for the values the machine uses and
{ref}`sec:simulation-resource-and-initialized-states` for the simulator's.
```